In [1]:
from utility.plot_spectrum import plot_spectrum
from utility.adjust_redshift import adjust_redshift
from utility.my_colors import pink

from sparcl.client import SparclClient
from dl import queryClient as qc

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [2]:
print(qc.schema('desi_dr1.agngal'))


Schema: desi_dr1
 Table: agngal

     Column Name   Description
     -----------   -----------
               z   Redshift from the FastSpecFit catalog
            zerr   Redshift error
       target_ra   Right Ascension in decimal degrees (J2000)
      target_dec   Declination in decimal degrees (J2000)
         min_mjd   MJD of the first exposure in coadded spectrum
        mean_mjd   Mean of MJD of exposures in a coadded spectrum
         max_mjd   MJD of the last exposure in coadded spectrum
            elon   Ecliptic longitude
            elat   Ecliptic latitude
            glon   Galactic longitude
            glat   Galactic latitude
        targetid   Unique identifier for each object observed by DESI
           zwarn   Warning flags (0 is good)
           ls_id   Legacy Surveys DR9 identifier from RELEASE, OBJID, BRICKID
     desi_target   DESI (dark time program) target selection bitmask
     scnd_target   SCND (secondary program) target selection bitmask
      bgs_target 

In [ ]:
# query = """
#         SELECT targetid, z, zerr, zwarn, civ_1549_flux, civ_1549_flux_ivar, civ_1549_sigma, flux_w3, flux_ivar_w3, spectype, survey
#         FROM desi_dr1.agngal
#         WHERE (z BETWEEN 2 AND 3.4) AND (zwarn=0) AND (civ_1549_flux>0) AND (spectype='QSO')
#         """

query = """
        SELECT targetid, z, zerr, zwarn, civ_1549_flux, civ_1549_flux_ivar, civ_1549_sigma, flux_w3, flux_ivar_w3, spectype, survey
        FROM desi_dr1.agngal
        WHERE (zwarn=0) AND (spectype='QSO') AND (z BETWEEN 2.2 AND 4.38) AND (civ_1549_flux > 0)
        """

In [11]:
results = qc.query(sql=query, fmt="pandas")

In [12]:
results.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404897 entries, 0 to 404896
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   targetid            404897 non-null  int64  
 1   z                   404897 non-null  float64
 2   zerr                404897 non-null  float64
 3   zwarn               404897 non-null  int64  
 4   civ_1549_flux       404897 non-null  float64
 5   civ_1549_flux_ivar  404897 non-null  float64
 6   civ_1549_sigma      404897 non-null  float64
 7   flux_w3             404897 non-null  float64
 8   flux_ivar_w3        404548 non-null  float64
 9   spectype            404897 non-null  object 
 10  survey              404897 non-null  object 
dtypes: float64(7), int64(2), object(2)
memory usage: 34.0+ MB


In [5]:
path = "data/local/query_results_from_dr1_agnqso.csv"
# results.to_csv(path)

results = pd.read_csv(path,index_col=0)
results.describe()

,targetid,z,zerr,zwarn,civ_1549_flux,civ_1549_flux_ivar,civ_1549_sigma,flux_w3,flux_ivar_w3,snr_w3
count,5.277990e+05,527799.000000,527799.000000,527799.0,5.277990e+05,527799.000000,527799.000000,5.277990e+05,5.273960e+05,527396.000000
mean,4.154664e+16,2.460149,0.000564,0.0,2.119578e+02,0.093695,2352.332128,3.772504e+01,1.401281e-03,1.887645
std,6.946310e+16,0.345008,0.000477,0.0,3.306140e+02,0.220041,1107.869556,5.920920e+04,3.924198e-03,9.689958
min,3.308235e+13,2.000003,0.000007,0.0,3.712253e-15,0.000138,1.234842,-3.394924e+07,6.138053e-12,-510.407105
25%,3.962778e+16,2.176329,0.000241,0.0,5.410338e+01,0.028505,1631.982900,-3.688999e+00,7.689273e-04,-0.124460
50%,3.962801e+16,2.384418,0.000434,0.0,1.176189e+02,0.052616,2308.604000,3.295674e+01,1.008971e-03,1.114798
75%,3.962848e+16,2.680793,0.000748,0.0,2.429008e+02,0.103262,2952.499250,7.996396e+01,1.468768e-03,2.666211
max,2.305843e+18,3.399991,0.023978,0.0,1.533291e+04,65.761260,9999.972000,2.564666e+07,2.832991e-01,1157.459851


In [12]:
# results["snr_w3"] = results["flux_w3"] * np.sqrt(results["flux_ivar_w3"])
results["civ_1549_fwhm"] = results["civ_1549_sigma"] * 2.355

In [14]:
results.query("snr_w3 > 3 & civ_1549_fwhm > 500 & civ_1549_fwhm < 20000")

,targetid,z,zerr,zwarn,civ_1549_flux,civ_1549_flux_ivar,civ_1549_sigma,flux_w3,flux_ivar_w3,spectype,survey,snr_w3,civ_1549_fwhm
5,39633221419271469,2.067840,0.000324,0,146.131410,0.011580,3203.6003,151.719940,0.003220,QSO,main,8.609555,7544.478707
9,39633217371767461,2.962281,0.000715,0,129.613590,0.048463,3775.2640,57.191692,0.003005,QSO,main,3.134989,8890.746720
16,39633221402494148,2.011882,0.001559,0,1.610962,0.013338,1798.4786,56.972410,0.003965,QSO,main,3.587633,4235.417103
17,39633221402494356,2.574769,0.000631,0,130.015520,0.031509,2685.2673,69.645035,0.003883,QSO,main,4.339632,6323.804492
18,39633221398303680,2.981374,0.000640,0,89.910680,0.066117,2263.1199,84.583810,0.004306,QSO,main,5.550246,5329.647365
...,...,...,...,...,...,...,...,...,...,...,...,...,...
527780,39633154419458822,2.341569,0.000190,0,649.471500,0.013979,2155.6748,115.410230,0.003073,QSO,main,6.397444,5076.614154
527781,39633154423653200,2.736704,0.000877,0,33.968170,0.220137,217.5175,205.015780,0.002997,QSO,main,11.223752,512.253713
527789,39633163047142707,2.211673,0.001106,0,91.779730,0.016726,2806.7650,102.519100,0.002832,QSO,main,5.455618,6609.931575
527793,39633163047144494,2.447516,0.000300,0,668.319030,0.011542,2872.7651,73.134560,0.002893,QSO,main,3.933513,6765.361811


In [19]:
# def adjust_z_lam(wavelength,z):
#     return wavelength/(1+z)
# adjust_z_lam(3600,3.4)

In [8]:
# client = SparclClient()

# idxs = results["targetid"].to_list()

# idxs = [int(x) for x in idxs]

# output_fields = ['specid', 'redshift', 'flux', 'wavelength', 'spectype', 'specprimary','survey', 'data_release', 'targetid', 'redshift_warning', 'desi_target']

In [9]:
# spectra = client.retrieve_by_specid(specid_list=idxs[:100], include=output_fields,dataset_list=["DESI-DR1"])

In [10]:
# spectra_df = pd.DataFrame(spectra.data[1:])
# spectra_df

In [11]:
# idx = 9


# lam = spectra_df["wavelength"][idx]
# flux = spectra_df["flux"][idx]
# z = spectra_df["redshift"][idx]

# lam, flux = adjust_redshift(lam, flux, z)

# plot_spectrum(lam, flux, line_lambda=1548, line_label="CIV")